In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("notebook-test").getOrCreate()
spark.range(5).show()
spark.stop()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [2]:
print(spark.sparkContext._jvm.java.lang.System.getProperty("java.version"))
print(spark.sparkContext._jvm.java.lang.System.getProperty("java.vendor"))

17.0.17
Eclipse Adoptium


In [3]:
# Create SparkSession (entry point for Spark)
spark = (
    SparkSession.builder
    .master("local[*]")              # Use all available CPU cores
    .appName("KMeans_Performance_mini-project")  # Name of the Spark application
    .config("spark.driver.memory", "4g")  # Allocate memory for Spark driver
    .getOrCreate()                   # Create or reuse SparkSession
)

In [4]:
from pathlib import Path
import pandas as pd

def find_repo_root(start: Path | None = None) -> Path:
    """
    Repo root is defined as the directory that contains `data/dataset.txt`
    """
    start = (start or Path.cwd()).resolve()

    for p in [start, *start.parents]:
        if (p / "data" / "dataset.txt").exists():
            return p

    raise FileNotFoundError(
        "Cannot find repo root. Expected `data/dataset.txt` in a parent directory.\n"
        "Have you run the data preparation script?"
    )

# 1) find repo root robustly
repo_root = find_repo_root()

# 2) read dataset pointer
pointer_file = repo_root / "data" / "dataset.txt"
dataset_dir = Path(pointer_file.read_text(encoding="utf-8").strip())


In [5]:
# 3) load data
csv_path = str(next(dataset_dir.glob("*.csv")))   # Spark ต้องการ string path

sdf = (spark.read
       .option("header", True)
       .option("inferSchema", True)
       .csv(csv_path))

sdf.show(5, truncate=False)
sdf.printSchema()

+---+-------+--------+-------------------+-------------------+-----------------+------------------+-------+-------+------------+-------------------------------------------------------------------------------------+-------------------------+------------+----------+-----+----------+-------+----------+------------+-------------------+--------------+-------------+-----------+------------+--------------+--------------+---------------+-----------------+-----------------+-------+-----+--------+--------+--------+-------+-------+----------+-------+-----+---------------+--------------+------------+--------------+--------------+-----------------+---------------------+
|ID |Source |Severity|Start_Time         |End_Time           |Start_Lat        |Start_Lng         |End_Lat|End_Lng|Distance(mi)|Description                                                                          |Street                   |City        |County    |State|Zipcode   |Country|Timezone  |Airport_Code|Weather_Timestamp  |T

In [8]:
sdf.head(), sdf.dtypes

(Row(ID='A-1', Source='Source2', Severity=3, Start_Time=datetime.datetime(2016, 2, 8, 5, 46), End_Time=datetime.datetime(2016, 2, 8, 11, 0), Start_Lat=39.865147, Start_Lng=-84.058723, End_Lat=None, End_Lng=None, Distance(mi)=0.01, Description='Right lane blocked due to accident on I-70 Eastbound at Exit 41 OH-235 State Route 4.', Street='I-70 E', City='Dayton', County='Montgomery', State='OH', Zipcode='45424', Country='US', Timezone='US/Eastern', Airport_Code='KFFO', Weather_Timestamp=datetime.datetime(2016, 2, 8, 5, 58), Temperature(F)=36.9, Wind_Chill(F)=None, Humidity(%)=91.0, Pressure(in)=29.68, Visibility(mi)=10.0, Wind_Direction='Calm', Wind_Speed(mph)=None, Precipitation(in)=0.02, Weather_Condition='Light Rain', Amenity=False, Bump=False, Crossing=False, Give_Way=False, Junction=False, No_Exit=False, Railway=False, Roundabout=False, Station=False, Stop=False, Traffic_Calming=False, Traffic_Signal=False, Turning_Loop=False, Sunrise_Sunset='Night', Civil_Twilight='Night', Nautical

In [11]:
# PySpark DataFrame doesn't have `.shape`. Use `.count()` for rows (expensive) and len(columns) for cols.
rows = sdf.count()  # this runs a Spark job on the dataset
cols = len(sdf.columns)
print(f"shape: ({rows}, {cols})")

shape: (7728394, 46)


In [6]:
from pyspark.sql.functions import col, sum

missing_df = sdf.select([sum(col(c).isNull().cast("int")).alias(c) for c in sdf.columns])
missing_df.show()

+---+------+--------+----------+--------+---------+---------+-------+-------+------------+-----------+------+----+------+-----+-------+-------+--------+------------+-----------------+--------------+-------------+-----------+------------+--------------+--------------+---------------+-----------------+-----------------+-------+----+--------+--------+--------+-------+-------+----------+-------+----+---------------+--------------+------------+--------------+--------------+-----------------+---------------------+
| ID|Source|Severity|Start_Time|End_Time|Start_Lat|Start_Lng|End_Lat|End_Lng|Distance(mi)|Description|Street|City|County|State|Zipcode|Country|Timezone|Airport_Code|Weather_Timestamp|Temperature(F)|Wind_Chill(F)|Humidity(%)|Pressure(in)|Visibility(mi)|Wind_Direction|Wind_Speed(mph)|Precipitation(in)|Weather_Condition|Amenity|Bump|Crossing|Give_Way|Junction|No_Exit|Railway|Roundabout|Station|Stop|Traffic_Calming|Traffic_Signal|Turning_Loop|Sunrise_Sunset|Civil_Twilight|Nautical_Twil

### Drop uneed columns


In [ ]:

sdf = sdf.drop('End_Lat', 'End_Lng', 'Wind_Chill(F)', 'Description', 'Zipcode', 'Airport_Code', 'Weather_Timestamp')

## TARGET VARIABLE ANALYSIS (SEVERITY)


In [9]:
from pyspark.sql.functions import col

rows = sdf.count()

severity_dist = (sdf.groupBy("Severity")
                 .count())
severity_dist = severity_dist.withColumn("percentage", (col("count") / rows) * 100)
severity_dist = severity_dist.sort("Severity")
severity_dist.show()

+--------+-------+------------------+
|Severity|  count|        percentage|
+--------+-------+------------------+
|       1|  67366|0.8716688098458748|
|       2|6156981|  79.6670174941909|
|       3|1299337| 16.81250981769304|
|       4| 204710|2.6488038782701815|
+--------+-------+------------------+



##  Class Imbalance Analysis:
• Most frequent class: 79.67%\
• Least frequent class: 0.87%\
• Imbalance ratio: 91.40:1\
 Dataset is imbalanced - consider sampling strategies

In [12]:
state_dist = (sdf.groupBy("State")
                 .count())

state_dist = state_dist.withColumn("percentage", (col("count") / rows) * 100)
state_dist = state_dist.sort("State")
state_dist.show()

+-----+-------+--------------------+
|State|  count|          percentage|
+-----+-------+--------------------+
|   AL| 101044|  1.3074385182743014|
|   AR|  22780|  0.2947572289922072|
|   AZ| 170609|  2.2075608464061225|
|   CA|1741433|  22.532922105161823|
|   CO|  90885|  1.1759881807268109|
|   CT|  71005|  0.9187549185509952|
|   DC|  18630| 0.24105913854806055|
|   DE|  14097| 0.18240529662436983|
|   FL| 880192|  11.389067379328745|
|   GA| 169234|  2.1897693104155924|
|   IA|  26307|  0.3403941362202807|
|   ID|  11376| 0.14719746431147274|
|   IL| 168958|   2.186198063918584|
|   IN|  67224|  0.8698314294017618|
|   KS|  20992|  0.2716217625550664|
|   KY|  32254| 0.41734414679168785|
|   LA| 149701|  1.9370259849588414|
|   MA|  61996|  0.8021847747410392|
|   MD| 140417|  1.8168975339507794|
|   ME|   2698|0.034910228438146396|
+-----+-------+--------------------+
only showing top 20 rows
